Imports

In [39]:
from xgboost import XGBRanker

from util import get_antisense

import pandas as pd
from IDO_Seq import IDO1_sequence
import requests

# import necessary feature modules
from scripts.data_genertion.consts import SEQUENCE
from scripts.features.feature_extraction import load_all_features
from scripts.data_genertion.data_handling import get_populated_df_with_structure_features

Load model

In [40]:
model = XGBRanker()
model.load_model('ranker_model2.json')

functions

In [41]:
def dna_to_dna_reverse_complement(seq: str) -> str:
    seq = seq.upper()
    translation_table = str.maketrans("ATGC", "TACG")
    # Translate and reverse
    return seq.translate(translation_table)[::-1]

In [42]:
def find_aso_binding_positions(aso_seq, mrna_seq):
    target = dna_to_dna_reverse_complement(aso_seq)
    idx = mrna_seq.find(target)
    if idx == -1:
        return None
    return int(idx), idx/len(mrna_seq)

Create gene_to_data

In [43]:
class TranscriptStructure:
    def __init__(self, transcript_id, full_mrna, exon_indices, intron_indices, utr_indices, cds_start):
        self.transcript_id = transcript_id
        self.full_mrna = full_mrna
        self.exon_indices = exon_indices
        self.intron_indices = intron_indices
        self.utr_indices = utr_indices
        self.cds_start = cds_start

    def __repr__(self):
        return (f"TranscriptStructure({self.transcript_id}, "
                f"{len(self.full_mrna)} nt, {len(self.exon_indices)} exons)")

In [44]:
def get_transcript_structure(transcript_id: str):
    server = "https://rest.ensembl.org"
    headers = {"Content-Type": "application/json"}

    # 1️⃣ Fetch transcript metadata (includes exons & UTRs)
    url = f"{server}/lookup/id/{transcript_id}?expand=1"
    r = requests.get(url, headers=headers)
    if not r.ok:
        raise RuntimeError(f"Failed to fetch transcript data: {r.text}")
    data = r.json()

    # 2️⃣ Fetch full pre-mRNA sequence
    seq_url = f"{server}/sequence/id/{transcript_id}?type=genomic"
    seq_r = requests.get(seq_url, headers=headers)
    if not seq_r.ok:
        raise RuntimeError(f"Failed to fetch sequence: {seq_r.text}")
    full_mrna = seq_r.json()["seq"]

    # 3️⃣ Extract exon coordinates relative to transcript
    exon_indices = []
    exons = sorted(data["Exon"], key=lambda e: e["start"])
    for e in exons:
        exon_indices.append((e["start"], e["end"]))

    # 4️⃣ Derive intron coordinates
    intron_indices = []
    for (prev_end, next_start) in zip([e["end"] for e in exons[:-1]],
                                      [e["start"] for e in exons[1:]]):
        intron_indices.append((prev_end + 1, next_start - 1))

    # 5️⃣ Extract CDS and UTR info
    cds_start = data.get("Translation", {}).get("start", None)
    utr_indices = []
    if "UTR" in data:
        utr_indices = [(u["start"], u["end"]) for u in data["UTR"]]

    # Create structure object
    return TranscriptStructure(
        transcript_id=transcript_id,
        full_mrna=full_mrna,
        exon_indices=exon_indices,
        intron_indices=intron_indices,
        utr_indices=utr_indices,
        cds_start=cds_start
    )


In [45]:
if __name__ == "__main__":
    transcript_id = "ENST00000518237"
    IDO1_structure_data_object = get_transcript_structure(transcript_id)

    print("✅ Successfully retrieved transcript structure!")
    print(IDO1_structure_data_object)

    # You can now store it like:
    genes_u = ["IDO1"]
    gene_to_data = {"IDO1": IDO1_structure_data_object}

✅ Successfully retrieved transcript structure!
TranscriptStructure(ENST00000518237, 14900 nt, 10 exons)


The data

In [46]:
target_gene = "IDO1"

exp_data = pd.read_csv("IDO1_exp_inhibition.csv")
exp_data = exp_data[["ASO", "Sequence", "Length"]]

exp_data[["sense_start", "normalized_start"]] = exp_data["Sequence"].apply(
    lambda s: pd.Series(find_aso_binding_positions(s, IDO1_sequence))
)
exp_data = exp_data.dropna(subset=["sense_start"])
exp_data["Cell line organism"] = "human"
exp_data["Canonical Gene Name"] = target_gene
exp_data["Inhibition(%)"] = 0.0  # dummy placeholder

exp_data

,ASO,Sequence,Length,sense_start,normalized_start,Cell line organism,Canonical Gene Name,Inhibition(%)
2,A06055H,CCGCAGGCCAGCATCAC,17,1002.0,0.541915,human,IDO1,0.0
3,A06049H,ACAAAACGTCCATGTTC,17,469.0,0.253651,human,IDO1,0.0
4,A06037H,CAGGACGTCAAAGCAC,16,844.0,0.456463,human,IDO1,0.0
5,A06017H,AGGACGTCAAAGCAC,15,844.0,0.456463,human,IDO1,0.0
6,A06048H,GTTGGCAGTAAGGAACA,17,355.0,0.191996,human,IDO1,0.0
...,...,...,...,...,...,...,...,...
69,A06001H,GGCGCTGTGACTTG,14,250.0,0.135208,human,IDO1,0.0
70,A06030H,AGGCGCTGTGACTTGT,16,249.0,0.134667,human,IDO1,0.0
71,A06045H,AGGCGCTGTGACTTGTG,17,248.0,0.134127,human,IDO1,0.0
72,A06029H,GGCGCTGTGACTTGTG,16,248.0,0.134127,human,IDO1,0.0


General variables

In [47]:
SEQUENCE = 'Sequence'
SENSE_START = 'sense_start'
SENSE_START_FROM_END = 'sense_start_from_end'
SENSE_LENGTH = 'sense_length'
SENSE_TYPE = 'sense_type'
SENSE_EXON = 'sense_exon'
SENSE_INTRON = 'sense_intron'
SENSE_UTR = 'sense_utr'
CELL_LINE_ORGANISM = 'Cell line organism'
INHIBITION = 'Inhibition(%)'
CANONICAL_GENE = 'Canonical Gene Name'

In [48]:
from asodesigner.util import get_antisense

def get_init_df(target_mrna: str, exp_data: pd.DataFrame) -> pd.DataFrame:
    """
    Initialize ASO DataFrame using sequences from exp_data instead of generating all possible candidates.

    Parameters
    ----------
    target_mrna : str
        Full mRNA sequence (5' → 3') of the target gene.
    exp_data : pd.DataFrame
        DataFrame containing at least a "Sequence" column with ASO sequences (DNA sense).

    Returns
    -------
    pd.DataFrame
        DataFrame with ASO features initialized: Sequence, sense_start, sense_length, sense_start_from_end.
    """
    sequences = exp_data["Sequence"].astype(str).tolist()
    sense_starts, sense_lengths, sense_starts_from_end = [], [], []

    for seq in sequences:
        # Find the sense binding position of this ASO
        pos_info = find_aso_binding_positions(seq, target_mrna)
        if pos_info is None:
            sense_starts.append(np.nan)
            sense_starts_from_end.append(np.nan)
        else:
            start_idx, normalized = pos_info
            sense_starts.append(start_idx)
            sense_starts_from_end.append(start_idx)

        sense_lengths.append(len(seq))

    df = pd.DataFrame({
        SEQUENCE: sequences,
        SENSE_START: sense_starts,
        SENSE_LENGTH: sense_lengths,
        SENSE_START_FROM_END: sense_starts_from_end
    })

    # Optionally merge back with other exp_data columns (like ASO name, Inhibition, etc.)
    if "ASO" in exp_data.columns:
        df["ASO"] = exp_data["ASO"].values
    if "Inhibition(%)" in exp_data.columns:
        df["Inhibition(%)"] = exp_data["Inhibition(%)"].values

    return df


# ✅ Example usage:
init_df = get_init_df(
    target_mrna=gene_to_data[target_gene].full_mrna,
    exp_data=exp_data
)


In [49]:
import numpy as np


def get_populated_df_with_structure_features(df, genes_u, gene_to_data):
    """
    Populate "the data" df with features like exon/intron, start of the sense strand, if found.
    """
    df_copy = df.copy()
    all_data_human = df_copy[df_copy[CELL_LINE_ORGANISM] == 'human']
    all_data_human_no_nan = all_data_human.dropna(subset=[INHIBITION]).copy()
    all_data_human_gene = all_data_human_no_nan[all_data_human_no_nan[CANONICAL_GENE].isin(genes_u)].copy()

    found = 0
    all_data_human_gene[SENSE_START] = np.zeros_like(all_data_human_gene[CANONICAL_GENE], dtype=int)
    all_data_human_gene[SENSE_START_FROM_END] = np.zeros_like(all_data_human_gene[CANONICAL_GENE], dtype=int)
    all_data_human_gene[SENSE_LENGTH] = np.zeros_like(all_data_human_gene[CANONICAL_GENE], dtype=int)
    all_data_human_gene[SENSE_EXON] = np.zeros_like(all_data_human_gene[CANONICAL_GENE], dtype=int)
    all_data_human_gene[SENSE_INTRON] = np.zeros_like(all_data_human_gene[CANONICAL_GENE], dtype=int)
    all_data_human_gene[SENSE_UTR] = np.zeros_like(all_data_human_gene[CANONICAL_GENE], dtype=int)
    all_data_human_gene[SENSE_TYPE] = "NA"
    for index, row in all_data_human_gene.iterrows():
        gene_name = row[CANONICAL_GENE]
        locus_info = gene_to_data[gene_name]
        pre_mrna = locus_info.full_mrna
        antisense = row[SEQUENCE]
        sense = get_antisense(antisense)
        idx = pre_mrna.find(sense)
        all_data_human_gene.loc[index, SENSE_START] = idx
        all_data_human_gene.loc[index, SENSE_START_FROM_END] = np.abs(
            locus_info.exon_indices[-1][1] - locus_info.cds_start - idx)
        all_data_human_gene.loc[index, SENSE_LENGTH] = len(antisense)
        if idx != -1:
            genome_corrected_index = idx + locus_info.cds_start
            found = False
            for exon_indices in locus_info.exon_indices:
                # print(exon[0], exon[1])
                if exon_indices[0] <= genome_corrected_index <= exon_indices[1]:
                    all_data_human_gene.loc[index, SENSE_TYPE] = 'exon'
                    all_data_human_gene.loc[index, SENSE_EXON] = 1
                    found = True
                    break
            for intron_indices in locus_info.intron_indices:
                # print(exon[0], exon[1])
                if intron_indices[0] <= genome_corrected_index <= intron_indices[1]:
                    all_data_human_gene.loc[index, SENSE_TYPE] = 'intron'
                    all_data_human_gene.loc[index, SENSE_INTRON] = 1
                    found = True
                    break
            for i, utr_indices in enumerate(locus_info.utr_indices):
                if utr_indices[0] <= genome_corrected_index <= utr_indices[1]:
                    all_data_human_gene.loc[index, SENSE_TYPE] = 'utr'
                    all_data_human_gene.loc[index, SENSE_UTR] = 1

                    found = True
                    break
        if not found:
            all_data_human_gene.loc[index, SENSE_TYPE] = 'intron'
    return all_data_human_gene

In [50]:
populated = get_populated_df_with_structure_features(exp_data, [target_gene], gene_to_data)

In [51]:
from scripts.data_genertion.data_handling import populate_features

VOLUME = 'ASO_volume(nM)'
TREATMENT_PERIOD = 'Treatment_Period(hours)'

target_mrna = gene_to_data[target_gene].full_mrna

populated[TREATMENT_PERIOD] = 24  # keep constant for all
populated[VOLUME] = 1000  # keep constant for all
populated['log_volume'] = np.log(populated[VOLUME])
populated['normalized_start'] = populated[SENSE_START] / len(target_mrna)
populated['normalized_sense_start_from_end'] = populated['sense_start_from_end'] / len(target_mrna)
easy_to_populate = ['at_skew', 'gc_content', 'gc_content_3_prime_5', 'gc_skew', 'hairpin_score',
                    'homooligo_count', 'internal_fold', 'nucleotide_diversity', 'self_energy', 'stop_codon_count',
                    'at_rich_region_score', 'poly_pyrimidine_stretch']
populate_features(populated, easy_to_populate)

In [52]:
import math
import ViennaRNA as RNA


def get_weighted_energy(target_start, l, step_size, energies, window_size):
    """
    Calculate average energy for a target region by finding which sliding windows
    overlap each position and averaging their energies.
    """
    if l <= 0 or len(energies) == 0:
        return 0.0

    num_windows = len(energies)
    position_energies = np.zeros(l, dtype=np.float64)

    for position in range(target_start, target_start + l):
        # Find all windows that overlap this position
        # A window at index k covers positions from k*step_size to k*step_size + window_size - 1

        # First window that could overlap: its end must reach this position
        first_window = max(0, math.ceil((position - window_size + 1) / step_size))

        # Last window that could overlap: its start must be at or before this position
        last_window = min(num_windows - 1, position // step_size)

        if first_window > last_window:
            # No windows fully overlap - use the closest window
            closest_window = np.clip(position // step_size, 0, num_windows - 1)
            energy_value = float(energies[closest_window])
        else:
            # Average all overlapping windows
            energy_value = float(np.mean(energies[first_window:last_window + 1], dtype=np.float64))

        position_energies[position - target_start] = energy_value

    return float(np.mean(position_energies, dtype=np.float64))


def calculate_energies(target_seq, step_size, window_size):
    L = len(target_seq)
    if L < window_size: return np.empty(0, dtype=np.float64)

    starts = list(range(0, L - window_size + 1, step_size))
    last_needed = L - window_size
    if starts[-1] != last_needed:
        starts.append(last_needed)  # add [L-window_size, L-1] window

    energies = np.empty(len(starts), dtype=np.float64)
    for k, i in enumerate(starts):
        _, mfe = RNA.fold(target_seq[i:i + window_size])
        energies[k] = float(mfe)
    return energies

In [53]:
def get_populate_fold(df, gene_to_data, fold_variants=[(40, 15)]):
    """Calculate RNA folding energies for each sequence in the dataframe."""
    from asodesigner.fold import calculate_energies, get_weighted_energy

    result_df = df.copy()

    for window_size, step_size in fold_variants:
        # Create column names for this window configuration
        fold_col = f'on_target_fold_openness{window_size}_{step_size}'
        norm_col = f'on_target_fold_openness_normalized{window_size}_{step_size}'

        # Initialize columns
        num_rows = len(result_df)
        result_df[fold_col] = np.zeros(num_rows, dtype=np.float64)
        result_df[norm_col] = np.zeros(num_rows, dtype=np.float64)

        # Cache energies for each gene to avoid recalculation
        gene_energies_cache = {}

        # Calculate weighted energy for each sequence region
        for idx, row in result_df.iterrows():
            target_gene = row[CANONICAL_GENE]

            # Calculate energies for this gene if not cached
            if target_gene not in gene_energies_cache:
                target_sequence = gene_to_data[target_gene].full_mrna
                gene_energies_cache[target_gene] = calculate_energies(
                    str(target_sequence), step_size, window_size
                )

            energies = gene_energies_cache[target_gene]

            length = row[SENSE_LENGTH]
            start_pos = row[SENSE_START]

            # Get average energy for this region
            avg_energy = get_weighted_energy(start_pos, length, step_size, energies, window_size)

            # Store raw and normalized values
            result_df.loc[idx, fold_col] = avg_energy
            result_df.loc[idx, norm_col] = avg_energy / length

    return result_df

In [54]:
fold_variants = [(40, 15)]
populated = get_populate_fold(populated, gene_to_data, fold_variants=fold_variants)

In [55]:
populated

,ASO,Sequence,Length,sense_start,normalized_start,Cell line organism,Canonical Gene Name,Inhibition(%),sense_start_from_end,sense_length,...,homooligo_count,hairpin_score,gc_skew,at_skew,nucleotide_diversity,stop_codon_count,at_rich_region_score,poly_pyrimidine_stretch,on_target_fold_openness40_15,on_target_fold_openness_normalized40_15
2,A06055H,CCGCAGGCCAGCATCAC,17,14053,0.943154,human,IDO1,0.0,814,17,...,0.000000,0.058824,-0.333333,0.600000,0.5625,0.000000,0.000000,0.0,-7.726471,-0.454498
3,A06049H,ACAAAACGTCCATGTTC,17,8661,0.581275,human,IDO1,0.0,6206,17,...,0.222222,0.117647,-0.428571,0.200000,0.6250,0.000000,0.058824,0.0,-7.125490,-0.419146
4,A06037H,CAGGACGTCAAAGCAC,16,11437,0.767584,human,IDO1,0.0,3430,16,...,0.176471,0.187500,-0.111111,0.714286,0.6250,0.000000,0.000000,0.0,-12.519792,-0.782487
5,A06017H,AGGACGTCAAAGCAC,15,11437,0.767584,human,IDO1,0.0,3430,15,...,0.187500,0.200000,0.000000,0.714286,0.6250,0.000000,0.000000,0.0,-12.478889,-0.831926
6,A06048H,GTTGGCAGTAAGGAACA,17,4944,0.331812,human,IDO1,0.0,9923,17,...,0.000000,0.000000,0.500000,0.333333,0.6875,0.058824,0.000000,0.0,-2.296078,-0.135063
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,A06001H,GGCGCTGTGACTTG,14,4232,0.284027,human,IDO1,0.0,10635,14,...,0.000000,0.071429,0.333333,-0.600000,0.5625,0.071429,0.000000,0.0,-4.389286,-0.313520
70,A06030H,AGGCGCTGTGACTTGT,16,4231,0.283960,human,IDO1,0.0,10636,16,...,0.000000,0.062500,0.333333,-0.428571,0.6250,0.062500,0.000000,0.0,-4.388542,-0.274284
71,A06045H,AGGCGCTGTGACTTGTG,17,4230,0.283893,human,IDO1,0.0,10637,17,...,0.000000,0.058824,0.400000,-0.428571,0.6250,0.058824,0.000000,0.0,-4.373529,-0.257266
72,A06029H,GGCGCTGTGACTTGTG,16,4230,0.283893,human,IDO1,0.0,10637,16,...,0.000000,0.062500,0.400000,-0.666667,0.5625,0.062500,0.000000,0.0,-4.357292,-0.272331


REMAINING FEATURES

Yehuda's feature - sense average accessibility

In [56]:
# sense_avg_accessibility (from RNA accessibility final parameters.ipynb)
from yehuda_code.access_calculator import AccessCalculator
import numpy as np

def compute_sense_accessibility(row, flank_size, access_win_size, seed_sizes, access_size,
                                min_gc=0, max_gc=100, gc_ranges=1):
    """
    Returns average accessibility for the ASO sense region based on AccessCalculator.calc output.
    Parameters used in the notebook: typical values are flank_size=120, access_win_size=70, seed_sizes=[13], access_size=??.
    Adjust parameters to match the notebook's final chosen values (see cell comments in the notebook).
    """
    try:
        if row['sense_start'] == -1 or pd.isna(row.get(f'sense_with_flank_{flank_size}nt')) or row.get(f'sense_with_flank_{flank_size}nt') == "":
            return np.nan

        seq = row[f'sense_with_flank_{flank_size}nt']
        sense_start = int(row['sense_start'])
        sense_length = int(row['sense_length'])

        # AccessCalculator.calc returns a DataFrame/structure with per-position accessibilities.
        # The notebook uses AccessCalculator.calc(seq, access_size, seed_sizes, ...) pattern.
        df_access = AccessCalculator.calc(seq, access_size, seed_sizes=seed_sizes,
                                          access_window=access_win_size,
                                          min_gc=min_gc, max_gc=max_gc, gc_ranges=gc_ranges)

        # from the notebook: take the region corresponding to the sense (relative to the flank sequence)
        # and compute an average accessibility (the exact column name depends on AccessCalculator)
        # Example: assume df_access has a numeric column 'accessibility' or per-pos columns.
        # The notebook computes mean across appropriate positions; here is a robust approach:

        # compute relative indices of sense within the flank sequence:
        sense_rel_start = sense_start  # if the flank sequence is aligned such that sense_start is the index
        sense_rel_end = sense_rel_start + sense_length

        # extract accessibility values for positions overlapping the sense
        # (the DataFrame layout can differ; adapt column selection if necessary)
        if hasattr(df_access, 'loc'):  # it's a DataFrame-like result
            # try commonly used column names; adapt if your AccessCalculator returns different columns
            # prefer 'mean_access' or 'accessibility' depending on module
            possible_cols = [c for c in df_access.columns if 'access' in c.lower() or 'prob' in c.lower()]
            if not possible_cols:
                # fallback: mean of numeric columns
                numeric_cols = df_access.select_dtypes(include=[float, int]).columns.tolist()
                values = df_access.loc[sense_rel_start:sense_rel_end-1, numeric_cols].mean(axis=1).values
            else:
                values = df_access.loc[sense_rel_start:sense_rel_end-1, possible_cols].mean(axis=1).values
            return float(np.nanmean(values))
        else:
            # if df_access is a list/array per position:
            arr = np.array(df_access)
            return float(np.nanmean(arr[sense_rel_start:sense_rel_end]))
    except Exception as e:
        # keep NaN if something fails for this row
        return np.nan

# --- Usage: choose the parameters used in the notebook ---
FLANK = 120
ACCESS_WIN = 70       # example: notebook saved win70
SEED_SIZES = [13]     # example seed size used in the notebook
ACCESS_SIZE = 70      # change to whatever AccessCalculator expects in the notebook

populated['sense_avg_accessibility'] = populated.apply(
    lambda r: compute_sense_accessibility(r, FLANK, ACCESS_WIN, SEED_SIZES, ACCESS_SIZE),
    axis=1
)


In [29]:
# RNaseH Krel computation (adapted from Test_RNaseH_features.ipynb)
# You must have the RNaseH processed data structure (the notebook uses rnaseh1_dict)
# Typical import / load (adjust path to where your team stored the processed RNaseH data):
# from scripts.rnaseh.prepare import load_rnaseh1_dict
# rnaseh1_dict = load_rnaseh1_dict()   # -> a dict keyed by experiment names (e.g. 'R7_krel') -> arrays of per-position scores

# If the notebook already saved a pickle/npz, load it, e.g.:
# import pickle
# with open("rnaseh1_dict.pkl", "rb") as fh:
#     rnaseh1_dict = pickle.load(fh)

start_positions = [0,1,2,3,4,5,6]  # notebook scanned these offsets
krel_experiments = ['R4a_krel','R4b_krel','R7_krel']  # experiments used in the notebook

# loop and compute feature columns
for exp in krel_experiments:
    for pos in start_positions:
        colname = f"RNaseH1_Krel_score_{exp}_start{pos}"
        # For each ASO we take the sum/mean/max of the precomputed scores overlapping the ASO window
        # The notebook often chooses the best window (max) per ASO. Here's a typical computation:
        def compute_krel_for_row(row, exp=exp, start_offset=pos):
            gene = row[CANONICAL_GENE]
            # target_mrna is the pre-mRNA sequence for that gene
            target_seq = gene_to_data[gene].full_mrna
            aso_len = int(row[SENSE_LENGTH])
            sense_start = int(row[SENSE_START])
            # window_start on transcript to look up in rnaseh1_dict depends on how the rnaseh1 scores were aligned;
            # the notebook uses precomputed per-position arrays aligned to full_mrna, so:
            scores_array = rnaseh1_dict[exp]  # array-like of scoring per base
            # The exact slicing depends on how 'pos' is meant to shift; this pattern matches the notebook:
            slice_start = sense_start + start_offset
            slice_end = slice_start + aso_len
            if slice_start < 0 or slice_end > len(scores_array):
                return float("nan")
            window_scores = scores_array[slice_start:slice_end]
            # aggregate: mean or sum or max — notebook chooses the best metric for each feature name.
            # For Krel they used a max or sum of krel over window; we return mean as safe default:
            return float(np.mean(window_scores))
        populated[colname] = populated.apply(compute_krel_for_row, axis=1)

# After you create the positional columns the notebook selects the "best" column per experiment
# e.g. create RNaseH1_Krel_score_R7_krel as the best (max) among the start positions:
populated['RNaseH1_Krel_score_R7_krel'] = populated[
    [f"RNaseH1_Krel_score_R7_krel_start{p}" for p in start_positions]
].max(axis=1)


NameError: name 'rnaseh1_dict' is not defined

In [31]:
from asodesigner.features.mod_features import compute_mod_min_distance_to_3prime

# the notebook used a column name like 'ChemistryPattern' or similar.
# In your pipeline earlier you set populated['Modification'] (string like 'MddddM' etc).
# Make sure to use the column the function expects (the notebook used "ChemistryPattern").

# If your modification string lives in populated['Modification'] use:
populated['Modification_min_distance_to_3prime'] = populated['Modification'].apply(
    lambda x: compute_mod_min_distance_to_3prime(x))

KeyError: 'Modification'

In [58]:
features = model.get_booster().feature_names
populated[features]

KeyError: "['CAI_score_global_CDS', 'RNaseH1_Krel_score_R7_krel', 'Modification_min_distance_to_3prime'] not in index"

In [60]:
populated.columns


Index(['ASO', 'Sequence', 'Length', 'sense_start', 'normalized_start',
       'Cell line organism', 'Canonical Gene Name', 'Inhibition(%)',
       'sense_start_from_end', 'sense_length', 'sense_exon', 'sense_intron',
       'sense_utr', 'sense_type', 'Treatment_Period(hours)', 'ASO_volume(nM)',
       'log_volume', 'normalized_sense_start_from_end', 'self_energy',
       'internal_fold', 'gc_content', 'gc_content_3_prime_5',
       'homooligo_count', 'hairpin_score', 'gc_skew', 'at_skew',
       'nucleotide_diversity', 'stop_codon_count', 'at_rich_region_score',
       'poly_pyrimidine_stretch', 'on_target_fold_openness40_15',
       'on_target_fold_openness_normalized40_15', 'sense_avg_accessibility'],
      dtype='object')